In [1]:
# Import des liberairies

#----Python
import os

# ----- Spark
from pyspark.sql.session import SparkSession
import pyspark.sql.functions as sf

#----Autres-------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- --Initialisation Spark ---
spark = (
    SparkSession.builder.appName('Projet')
    .config('spark.driver.memory', '4g') # autoriser le driver d'utiliser jusqu'à 4gb
    .config('spark.executer.memory','4g')
    .config('spark.sql.shuffle.partitions','50') # par defaut c'est 200, 50 pour réduire la tension sur la machine
    .config('spark.sql.execution.arrow.pyspark.enabaled' ,'true') # pour une transition rapide entre pandas et spark df
    .getOrCreate()
)

# --- Formatage du texte ---
# character Gras
B = '\033[1m' 
BE = '\033[0m'

#Chemin vers les données sur ma machine
CHEMIN_INTERIM = '../data/interim'

In [2]:
df_Feat = spark.read.parquet(CHEMIN_INTERIM + '/df_Feat.parquet')
print(f"{df_Feat.count():,} lignes, {len(df_Feat.columns)} colonnes")
df_Feat.printSchema()

2,024,836 lignes, 43 colonnes
root
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- A3_Temperature: double (nullable = true)
 |-- A3_pH: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_NO2: double (nullable = true)
 |-- A3_NO3: double (nullable = true)
 |-- A3_PT: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- A3_Volume: double (nullable = true)
 |-- A3_Pluie: double (nullable = true)
 |-- A4_Temperature: double (nullable = true)
 |-- A4_pH: double (nullable = true)
 |-- A4_MES: double (nulla

In [3]:
# Les seuils par paramètre et catégorie
# Selon l'arrêté du 21 juillet 2015 (cas général, zone sensible — bassin Seine-Normandie) :
# Paramètre    | <2000 EH | 2000-10000 EH | > 10000 EH | 
# DBO5 (mg/L)  |≤ 35      |≤ 25           |≤ 25        |
# DCO (mg/L)   |≤ 200     |≤ 125          |≤ 125       |
# MES (mg/L)   | —        |≤ 35           |≤ 35        |
# NTK (mg/L)   | —        |—              |≤ 15        |
# PT (mg/L)    | —        |—              |≤ 2         |

In [4]:
# Construction du label Conforme à partir des seuils ERU (arrêté 21/07/2015)

# Helper : "param respecte le seuil" = (valeur ≤ seuil) OU (valeur NULL = pas mesuré, donc on suppose OK)
def respecte(col, seuil):
    return (sf.col(col).isNull()) | (sf.col(col)<=seuil)

df_LabelConf = (
    df_Feat
    .withColumn(
        'ConfPar'
        ,sf.when(sf.col('CatSta')=='petite'
                 ,(respecte('A4_DBO5', 35) & respecte('A4_DCO', 200)).cast('int'))
        .when(sf.col('CatSta')=='moyenne'
                 ,(respecte('A4_DBO5', 25) & respecte('A4_DCO', 125) & respecte('A4_MES', 35)).cast('int'))
        .when(sf.col('CatSta')=='grande'
                 ,(respecte('A4_DBO5', 25) & respecte('A4_DCO', 125) & respecte('A4_MES', 35) & respecte('A4_NTK', 15) & respecte('A4_PT', 2)).cast('int'))
    )
)

# Vérification
df_LabelConf.groupBy("CatSta", "ConfPar").count().orderBy("CatSta", "ConfPar").show()

+-------+-------+------+
| CatSta|ConfPar| count|
+-------+-------+------+
| grande|      0|  3410|
| grande|      1|375270|
|moyenne|      0|   679|
|moyenne|      1|828690|
| petite|      0|   502|
| petite|      1|816285|
+-------+-------+------+



In [5]:
# On filtre sur les lignes où au moins DBO5 ET DCO ET MES sont renseignés en A4 
# (les 3 paramètres clés mesurés à chaque vrai contrôle réglementaire) 
df_LabelConf_Net = df_LabelConf.filter(
    sf.col("A4_DBO5").isNotNull() 
    & sf.col("A4_DCO").isNotNull() 
    & sf.col("A4_MES").isNotNull()
)

print(f"Avant : {df_LabelConf.count():,}")
print(f"Après : {df_LabelConf_Net.count():,}")

df_LabelConf_Net.groupBy("CatSta", "ConfPar").count().orderBy("CatSta", "ConfPar").show()

Avant : 2,024,836


Après : 92,840


+-------+-------+-----+
| CatSta|ConfPar|count|
+-------+-------+-----+
| grande|      0| 2970|
| grande|      1|53246|
|moyenne|      0|  676|
|moyenne|      1|27343|
| petite|      0|  426|
| petite|      1| 8179|
+-------+-------+-----+



In [6]:
# df_LabelConf_Net.write.mode('overwrite').parquet(CHEMIN_INTERIM + '/df_Label.parquet')
# print("dataframe sauvegardé en Parquet")

In [7]:
# sauvgarde
df_LabelConf_Net.write.mode('overwrite').parquet(CHEMIN_INTERIM + '/df_LabelConf.parquet')
print('dataframe sauvgardé en parquet')

dataframe sauvgardé en parquet


In [8]:
spark.stop()